# 05. YOLO 검출과 사람 Pose / 90분

**강사 설명 → 완성된 셀 실행 → 조건 비교 → 결과 해설** 순서입니다. 코드 작성 과제나 관찰 답안 제출은 없습니다.

검출 박스는 영역이고 pose 관절은 제어할 지점입니다. 박스 점수와 관절 점수는 같은 값이 아닙니다.

[단원 실행 안내](README.md) · [슬라이드 실습 시작](https://docs.google.com/presentation/d/1J5Jw6Lg-2oYFKec-RROYlt6nxtG-IHVHd7hLwH8gd9s/edit#slide=id.d4_lab05_start)

각 단원은 새 커널에서 시작합니다. 노트북 셀은 위에서 아래로 실행하고, `bash` 명령은 지정한 터미널에서 실행합니다. 결과가 이미 있으면 재사용 여부와 실행 이름을 확인합니다.


## 1. 아키텍처 — 지금 실행하는 부분

![YOLO 검출과 사람 Pose의 입력·처리·출력](../교재그림/guided-vla/day4-05.svg)

filter_detections는 class와 신뢰도를 검사합니다. pose 경로는 모델이 제공한 사람 박스와 관절을 people_from_yolo에서 정규화 자료로 바꿉니다. select_largest_person은 유효 후보 중 가장 큰 면적을 선택하고, pose_observation은 누락 관절까지 포함해 17개 이름을 유지합니다.

**그림을 읽는 순서:** 1→2→3 뒤 아래 4→5→6으로 이어집니다. 실시간 반복은 새 관측이 다음 입력으로 들어오는 과정이며, 저장 결과를 읽는 셀은 모델을 다시 실행하지 않습니다.


### 전체 실행 지도에서 현재 단원 찾기

![4일차 실행 경계 전체 지도](../교재그림/guided-vla/detailed/day4-system.svg)

터미널 A/B/C는 실행·관찰 역할이고 ROS 노드와 HTTP 서비스는 실제 처리 주체입니다. 02는 파일 요청만, 05는 독립 YOLO 실행만 다루므로 매 단원에서 모든 상자를 동시에 실행하지 않습니다. 아래 단원별 도식과 함께 현재 경로를 확인합니다.

## 상세 해설 — 앞선 수업 없이도 흐름을 읽기

아래 설명은 1~3일차에서 배운 개념도 이번 실습의 입력·코드·출력에 맞춰 다시 풉니다. 실행 중에는 필요한 부분을 읽고, 수업 뒤에는 독립적인 참고 자료로 사용할 수 있습니다. 추가 작성 과제는 없습니다.

### 1. 기초 복습: 검출 박스와 관절의 정보량

- detection은 class·box·confidence를 반환합니다.
- pose는 사람 박스와 17개 관절을 반환합니다.
- 박스 confidence와 관절 confidence는 별도 값입니다.
- 픽셀 xyxy와 정규화 xy를 구분해야 합니다.

객체 검출의 bbox=[x1,y1,x2,y2]는 대상이 있는 사각 영역입니다. 박스 중심이 코나 손목의 위치라는 뜻은 아닙니다. pose 모델은 사람의 관절을 이름에 대응하는 순서로 예측하므로 특정 신체 지점을 제어 목표로 선택할 수 있습니다. detection과 pose는 각각의 모델 실행이며 detection 출력을 pose 모델에 반드시 순차 입력하는 파이프라인으로 구현된 것은 아닙니다.

confidence는 모델이 제공한 점수입니다. 0.8이라는 숫자를 이 장면에서 80% 확률로 맞다는 보정된 확률로 단정하지 않습니다. 사람 박스 점수가 높아도 가려진 손목의 점수는 낮을 수 있습니다. 따라서 사람 선택 임계값과 목표 관절 임계값을 별도로 둡니다.

일반 detection 계약의 bbox_xyxy는 영상 픽셀 기준이고, people_from_yolo의 boxes.xyxyn과 keypoints.xyn은 정규화 좌표입니다. 한쪽의 100이라는 픽셀 좌표를 다른 쪽의 0~1 계약에 넣으면 범위 검사에서 거부됩니다. 배열 길이뿐 아니라 단위와 좌표 기준을 확인해야 합니다.


### 2. 실행 아키텍처: 모델 출력에서 한 사람 관측까지

![실행 주체와 데이터 경계](../교재그림/guided-vla/detailed/day4-05.svg)

- 원본 BGR → YOLO pose Results
- people_from_yolo: CPU 목록·관절 이름으로 변환
- select_largest_person: 신뢰도 필터 후 면적 비교
- pose_observation: bbox와 이름 붙은 17관절 보존

YOLO Results는 tensor와 모델별 속성을 포함합니다. people_from_yolo는 boxes와 keypoints를 detach().cpu().tolist()로 일반 Python 자료로 옮기고 class 이름이 person인 후보만 모읍니다. 이는 모델 실행과 이후 JSON 검증·제어를 분리하는 어댑터 경계입니다. GPU tensor를 JSON에 그대로 넣을 수 없다는 점도 여기서 해결합니다.

select_largest_person은 confidence와 box의 유효성을 먼저 검사한 후 면적이 가장 큰 후보를 고릅니다. 점수가 낮은 큰 박스를 먼저 선택하고 나중에 버리는 순서와 결과가 다릅니다. 같은 면적이면 먼저 나온 후보가 유지되며 결과는 원본 입력과 독립된 복사입니다.

pose_observation은 선택된 사람의 17개 이름을 모두 유지합니다. 관절이 없거나 잘못되면 xy=None, confidence=0.0으로 표시합니다. 리스트에서 관절을 삭제해 뒤 인덱스를 당기지 않으므로 nose와 left_wrist의 의미가 바뀌지 않습니다. 사람이 없으면 person=None으로 후속 단계에 명확히 전달합니다.


### 3. 코드 추적: 필터 순서와 누락 처리

- 0~1 범위·양의 box 면적·threshold를 검사합니다.
- 유효 후보만 area=(x2−x1)(y2−y1)로 비교합니다.
- 관절 신뢰도가 없으면 좌표가 있어도 confidence=0입니다.
- 최대 면적 선택은 매 frame 수행하며 신원 추적이 아닙니다.

합성 예제에서 confidence 0.6인 큰 사람과 0.9인 작은 사람이 있다고 가정하면 threshold=0.5에서는 큰 사람을, 0.8에서는 작은 사람을 선택할 수 있습니다. 바뀐 것은 사람이 갑자기 작아진 것이 아니라 후보 집합입니다. 수업 셀의 검출 개수 비교와 사람 선택 비교는 이 필터 순서를 읽는 예제입니다.

people_from_yolo는 keypoints.conf가 없으면 관절 confidence를 0으로 채웁니다. 좌표 숫자가 존재한다는 이유만으로 근거 있는 지점으로 승인하지 않기 위해서입니다. pose_observation에서는 누락 좌표를 [0,0]으로 대신하지 않습니다. [0,0]은 영상 모서리의 실제 위치를 뜻할 수 있어 누락과 구분되지 않기 때문입니다.

최대 면적은 보통 화면에서 큰 사람을 선택하는 간단한 기준입니다. 사람의 고유 ID를 유지하지 않으므로 두 사람이 앞뒤로 움직이면 선택 대상이 바뀝니다. 동일 인물을 계속 따라가려면 별도의 추적 정책과 소실·재선택 규칙이 필요합니다.


### 4. 실행 화면과 수정할 경계

- 박스 수·선택 박스·관절 점을 서로 구분해 봅니다.
- person_conf 변경은 후보, keypoint_conf는 목표 승인에 영향
- 카메라 단독 실행 전 ROS 카메라를 종료합니다.
- 모델 교체 시 클래스·관절 순서·정규화 계약을 확인합니다.

실시간 화면에서 전체 박스와 선택된 사람의 표시를 구분합니다. 사람이 검출되는데 코가 표시되지 않는다면 전체 검출 실패가 아니라 관절 가림이나 신뢰도 문제일 수 있습니다. threshold를 높여 화면이 깨끗해져도 실제 대상이 누락될 수 있으므로 검출 수가 줄었다는 사실 자체를 정확도 개선으로 표현하지 않습니다.

05의 독립 live_yolo.py --source camera는 카메라 장치를 직접 엽니다. 이전 ROS 카메라가 장치를 잡고 있으면 단독 실행과 충돌할 수 있으므로 기존 카메라와 제어기를 종료한 뒤 실행합니다. 06으로 갈 때는 단독 YOLO를 종료하고 ROS 카메라를 다시 시작합니다. 화면에 창이 여러 개 떠 있다는 사실보다 누가 실제 장치를 읽는지 확인하는 것이 중요합니다.

직접 모델을 바꾼다면 people_from_yolo의 Results 읽기와 COCO_KEYPOINTS 순서를 확인합니다. 관절 수가 같아도 순서가 다르면 코라고 표시한 좌표가 다른 관절일 수 있습니다. 표시 좌표와 제어 좌표의 미러링도 분리해서 유지해야 합니다.


## 2. 코드 위치와 변경 범위

| 구분 | 이 단원에서 확인할 내용 |
|---|---|
| 실제 소스 | [ros_ws/src/nepes_vla_demo/nepes_vla_core/pose_contracts.py](ros_ws/src/nepes_vla_demo/nepes_vla_core/pose_contracts.py) |
| 바꾸어 볼 조건 | `--confidence 0.5 / 0.8` |
| 예상되는 차이 | 임계값 0.5와 0.8에서 남는 박스 수를 비교합니다. 면적이 더 크지만 신뢰도가 낮은 사람은 제외되는지 확인한 뒤 실제 두 사람의 거리 변화로 선택 전환을 봅니다. |
| 직접 구현·수정할 때 | 신뢰도 필터를 면적 비교보다 먼저 적용합니다. bbox 픽셀 좌표와 pose 정규화 좌표를 혼용하지 않습니다. 최대 면적 선택은 신원 추적이 아니므로 다른 사람이 가까워지면 선택이 바뀝니다. |

설정값 변경은 아래 준비된 비교 실행으로 진행합니다. 소스를 직접 변경하는 경우에는 실행 중인 프로세스를 종료하고 다시 실행해야 합니다. ROS 패키지 파일은 재빌드·환경 source까지 확인합니다.


## 3. 준비 — 실제 실행 코드와 같은 함수를 사용합니다

아래 CPU 예제는 카메라·모델·ROS 없이 자료의 흐름을 분리해 보여 줍니다. 통과가 실제 인식 성공이나 장비 성능을 뜻하지는 않습니다.


In [ ]:
from pathlib import Path
import sys, json, inspect, uuid
from datetime import datetime, timezone
from IPython.display import display, Markdown, Code
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'09.Pseudo-VLA_v2').is_dir())
LESSON = ROOT/'09.Pseudo-VLA_v2'
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
# 매 단원 새 커널: 다른 일차의 common 모듈이 남아 있지 않게 합니다.
RUN = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
OUT = ROOT/'Data/outputs/09.Pseudo-VLA_v2/guided'/RUN

# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
def show_rows(rows):
    """조건·결과를 한 표에서 읽습니다. 수강생이 답안 dict를 채우지 않습니다."""
    if not rows:
        print('표시할 결과 없음')
        return
    keys = list(dict.fromkeys(key for row in rows for key in row))
    def value(v):
        return str(v).replace('|', '/').replace('\n', ' ')
    lines = [' | '.join(keys), ' | '.join(['---']*len(keys))]
    lines += [' | '.join(value(row.get(key, '')) for key in keys) for row in rows]
    display(Markdown('\n'.join(lines)))

# 비교 출력을 자동 저장합니다. 수강생이 관찰 답안을 작성하거나 수치를 옮겨 적지 않습니다.
def save_demo(name, rows):
    """합성 예제임을 파일에도 명시합니다. 모델 측정값이나 수강생 답안이 아닙니다."""
    OUT.mkdir(parents=True, exist_ok=True)
    path = OUT/(name+'.json')
    path.write_text(json.dumps({'kind':'synthetic-guided-demo', 'model_measured':False,
                               'rows':rows}, ensure_ascii=False, indent=2)+'\n')
    print('자동 저장:', path)
print('실행 폴더:', LESSON)


### 완성 코드 읽기

아래 셀은 실행에 쓰는 함수 본문을 그대로 표시합니다. **입력 검사 → 조건 분기 → 결과 반환**을 순서대로 읽습니다. 노트북에 복사한 별도 정답과 실제 동작이 어긋나지 않도록 같은 소스를 조회합니다.


In [ ]:
from common.pose_contracts import select_largest_person, pose_observation
# 함수의 실제 파일과 주석을 함께 표시합니다. 이 셀은 소스를 수정하지 않습니다.
for function in (select_largest_person, pose_observation,):
    print(inspect.getsourcefile(function))
    display(Code(inspect.getsource(function), language='python'))


### 한 줄씩 읽는 처리 과정

filter_detections는 class와 신뢰도를 검사합니다. pose 경로는 모델이 제공한 사람 박스와 관절을 people_from_yolo에서 정규화 자료로 바꿉니다. select_largest_person은 유효 후보 중 가장 큰 면적을 선택하고, pose_observation은 누락 관절까지 포함해 17개 이름을 유지합니다.

신뢰도 필터를 면적 비교보다 먼저 적용합니다. bbox 픽셀 좌표와 pose 정규화 좌표를 혼용하지 않습니다. 최대 면적 선택은 신원 추적이 아니므로 다른 사람이 가까워지면 선택이 바뀝니다.


## 4. 조건을 바꾸어 실행하고 표로 비교하기

임계값 0.5와 0.8에서 남는 박스 수를 비교합니다. 면적이 더 크지만 신뢰도가 낮은 사람은 제외되는지 확인한 뒤 실제 두 사람의 거리 변화로 선택 전환을 봅니다.

아래 입력은 경계 조건을 설명하기 위한 고정 예제입니다. 주석을 읽고 셀 전체를 실행하면 비교표와 검사 결과가 자동으로 나옵니다.


In [ ]:
from common.contracts import filter_detections
from common.pose_contracts import select_largest_person, pose_observation

# 고정된 검출을 사용해 threshold 효과를 분리합니다. 실제 인식률을 측정하지 않습니다.
records = [dict(label='person', confidence=score, bbox_xyxy=[0, 0, 60, 70],
                frame_id=f'd{i}', captured_ms=1000) for i, score in enumerate((.55, .75, .95))]
rows = []
for threshold in (.5, .8):
    selected = filter_detections(records, allowed_labels={'person'}, min_confidence=threshold,
                                 image_width=100, image_height=80)
    rows.append(dict(threshold=threshold, count=len(selected), scores=[x['confidence'] for x in selected]))
# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
show_rows(rows)
assert [r['count'] for r in rows] == [3, 1]

# 면적은 B가 더 크지만 confidence=.6이므로 threshold=.8에서는 A가 선택됩니다.
people = [dict(bbox=[.1,.1,.4,.6], confidence=.95, keypoints={}),
          dict(bbox=[.1,.1,.9,.9], confidence=.6, keypoints={})]
pose_rows = []
for threshold in (.5, .8):
    selected = select_largest_person(people, min_conf=threshold)
    obs = pose_observation(people, 'pose-demo', 1000, min_conf=threshold)
    pose_rows.append(dict(threshold=threshold, bbox=selected['bbox'], confidence=selected['confidence'],
                          joint_count=len(obs['person']['keypoints'])))
show_rows(pose_rows)
assert [r['confidence'] for r in pose_rows] == [.6, .95]
assert all(r['joint_count'] == 17 for r in pose_rows)
# 비교 출력을 자동 저장합니다. 수강생이 관찰 답안을 작성하거나 수치를 옮겨 적지 않습니다.
save_demo('detection-pose-selection', rows + pose_rows)


## 5. 실제 입력 실행 — 화면과 로그 연결

본수업은 로컬 카메라 화면을 함께 보며 역할을 교대합니다. 아래의 A/B/C는 터미널 역할입니다. 카메라와 제어기는 각각 하나만 실행합니다. 실제 실행의 지연과 인식 결과는 위 합성 예제와 별도로 확인합니다.


> **구분: 수업 핵심** — 강사 설명을 들은 뒤 수강생 실습 구간에서 진행합니다.

## 3. 실제 YOLO 경로

```bash
bash
python /workspace/09.Pseudo-VLA_v2/scripts/live_yolo.py   --task detect   --model /workspace/Data/models/yolo26n.pt   --source /workspace/Data/images/road.png
```


### USB 웹캠 — 촬영하면서 결과 보기

**Jetson에 연결된 USB 웹캠을 로컬 사용자 한 명만 실행합니다.** 원격 사용자는 같은 카메라를 중복 실행하지 않고 로컬 화면을 함께 봅니다. 앞서 실행한 MediaPipe·ROS 카메라 노드 등 카메라를 사용하는 프로그램은 먼저 종료하세요.

로컬 Jetson 화면에서 사용하는 **실습 컨테이너 내부 터미널** 에 아래 명령을 붙여 넣습니다. 현재 폴더와 관계없이 실행할 수 있도록 절대 경로를 사용합니다. `--camera-index 0`은 `/dev/video0`이며, 다른 장치로 연결되었다면 번호를 바꿉니다.

```bash
bash
python /workspace/09.Pseudo-VLA_v2/scripts/live_yolo.py --task detect --model /workspace/Data/models/yolo26n.pt --source camera --camera-index 0 --preview --max-frames 300
```

**입력 → 처리 → 화면:** USB 웹캠 프레임 → YOLO `detect` 추론 → 사각형 상자·객체 이름·confidence(검출 점수)를 원본 영상 위에 표시합니다. 카메라 앞에 사람이나 모델이 학습한 물체를 놓고 거리·조명을 바꾸며 결과를 비교하세요. 모든 물체 종류를 인식하는 것은 아니며, 검출이 없으면 원본 영상만 표시될 수 있습니다. 좌표는 좌우 반전하지 않은 입력 영상 기준입니다.

- **종료:** 결과 창에 포커스를 두고 `q` 또는 `Esc`를 누릅니다. 터미널에서 `Ctrl+C`로 끝낼 수도 있습니다. 최대 300프레임을 처리하면 자동 종료하며, 300초를 뜻하지 않습니다.
- **모델 선택:** 검출은 `yolo26n.pt`, 뒤 Pose 실습은 `yolo26n-pose.pt`를 사용합니다. `yolo26n-seg.pt`는 07 선택 자습용입니다. task와 모델이 다르면 오류로 안내합니다.
- **저장:** 종료 시 마지막으로 처리가 완료된 프레임을 `Data/outputs/09.Pseudo-VLA_v2/yolo/detect-camera-last.jpg`와 같은 이름의 `.json`에 저장합니다. 재실행하면 덮어쓰며 전체 동영상은 저장하지 않습니다. JSON의 `frame_id`, `captured_ms`, 검출 좌표·점수를 이미지와 함께 확인합니다.
- **화면·연결 문제:** 표시 창은 브라우저가 아니라 로컬 Jetson 데스크톱에 열립니다. `DISPLAY` 오류는 로컬 그래픽 터미널과 컨테이너의 화면 연결을, 카메라 열기 실패는 USB 연결·장치 번호·컨테이너 장치 접근·다른 프로그램의 점유를 확인하세요.

**종료 후 노트북에서 마지막 결과 보기:** 아래 제공 코드를 실행합니다. 저장된 이미지만 읽으며 카메라나 모델을 다시 실행하지 않습니다. 파일이 없다면 카메라 실행에서 최소 한 프레임이 완료되었는지 확인합니다. 이전 실행 파일일 수도 있으므로 JSON의 캡처 시각을 함께 확인하세요.

```python
from pathlib import Path
from IPython.display import Image, display
import json

result_dir = ROOT / "Data/outputs/09.Pseudo-VLA_v2/yolo"
image_path = result_dir / "detect-camera-last.jpg"
json_path = result_dir / "detect-camera-last.json"
if image_path.is_file() and json_path.is_file():
    display(Image(filename=str(image_path)))
    print(json.dumps(json.loads(json_path.read_text()), ensure_ascii=False, indent=2))
else:
    print("먼저 위 USB 웹캠 명령으로 한 프레임 이상 처리한 뒤 종료하세요.")
```


## 6. 실제 Pose 확인 — 17개 관절과 선택된 사람

모델은 수업 전에 `/workspace/Data/models/yolo26n-pose.pt`에 준비합니다. 이미지를 네트워크로 보내는 단계는 아직 없습니다.

**정지 사진:** 새 JupyterLab 터미널에서 실행합니다. 앞선 ROS 제어기 B는 종료하고, 사진 입력은 카메라를 점유하지 않습니다.
```bash
bash
python /workspace/09.Pseudo-VLA_v2/scripts/live_yolo.py --task pose --model /workspace/Data/models/yolo26n-pose.pt --source /workspace/Data/images/07.Mediapipe/person_open_palm.png
```

**USB 웹캠:** 앞선 ROS 카메라 A와 detection 카메라 프로그램을 먼저 종료합니다. 카메라를 사용하는 프로그램은 하나만 실행합니다. 새 JupyterLab 터미널에서 아래 전체를 실행하고, 종료는 영상 창의 `q`/`Esc` 또는 터미널의 Ctrl+C입니다.
```bash
bash
python /workspace/09.Pseudo-VLA_v2/scripts/live_yolo.py --task pose --model /workspace/Data/models/yolo26n-pose.pt --source camera --camera-index 0 --preview --max-frames 300
```

다수 인물·코 가림·얼굴이 영상 가장자리인 조건을 비교합니다. 종료 후 마지막 사진은 `Data/outputs/09.Pseudo-VLA_v2/yolo/pose-camera-last.jpg`, JSON은 같은 이름의 `.json`입니다. `people` 전체 목록과 `observation.person`에 선택된 사람을 확인합니다. `selected` 박스·관절을 화면과 대조하세요. 관절 좌표는 정규화 값이고 박스·관절 confidence는 서로 다른 값입니다. 재실행하면 마지막 결과 파일을 덮어씁니다.


### 같은 영상으로 confidence 비교

카메라 A와 독립 YOLO 창을 종료하고, 아래 두 명령을 **차례로** 실행합니다. 입력을 고정해 threshold 하나만 비교하며 별도 폴더로 저장합니다.
```bash
python /workspace/09.Pseudo-VLA_v2/scripts/live_yolo.py --task detect --model /workspace/Data/models/yolo26n.pt --source /workspace/Data/images/07.Mediapipe/person_open_palm.png --confidence 0.5 --output-dir /workspace/Data/outputs/09.Pseudo-VLA_v2/yolo/conf05
python /workspace/09.Pseudo-VLA_v2/scripts/live_yolo.py --task detect --model /workspace/Data/models/yolo26n.pt --source /workspace/Data/images/07.Mediapipe/person_open_palm.png --confidence 0.8 --output-dir /workspace/Data/outputs/09.Pseudo-VLA_v2/yolo/conf08
```
검출 수가 같을 수도 있습니다. 임계값 사이에 점수가 있는 박스가 없으면 정상적으로 같은 결과가 나옵니다. 06에서는 독립 카메라 프로그램을 종료하고 ROS 카메라 A를 다시 시작합니다.


## 6. 결과 해설과 다음 단계

검출 박스는 영역이고 pose 관절은 제어할 지점입니다. 박스 점수와 관절 점수는 같은 값이 아닙니다.

임계값 0.5와 0.8에서 남는 박스 수를 비교합니다. 면적이 더 크지만 신뢰도가 낮은 사람은 제외되는지 확인한 뒤 실제 두 사람의 거리 변화로 선택 전환을 봅니다.

이제 이미지 대신 선택된 사람의 전체 관절 정보를 LLM에 주고, 언어로 사용할 관절을 정합니다.

비교표와 실제 화면·로그를 강사 설명과 대조합니다. 별도 답안 작성은 없으며, 자동 저장 파일과 실행 결과로 진행 상태를 확인합니다.
